# 05 · QLoRA: LoRA on a 4-bit base, with bitsandbytes + PEFT

QLoRA is LoRA with the frozen base stored in 4 bits. Here you load SmolLM2-135M-Instruct in 4-bit NF4 with double
quantization (on the CPU — bitsandbytes supports it in this environment), prepare it for training, attach LoRA,
train a few steps with a paged 8-bit optimizer, and do the memory arithmetic that decides LoRA vs QLoRA for a 7B model.

**Time:** ~30 min · **Runs:** offline on CPU in ~1–2 min (uses the cached SmolLM2 model) · **Needs:**
[04 · LoRA with PEFT](04_lora_with_peft.ipynb), [Quantization with bitsandbytes](../10_huggingface/06_quantization_with_bitsandbytes.ipynb),
[Quantization int8 / int4 / NF4](../07_genai_foundations/07_quantization_int8_int4_nf4.ipynb)

## Why this matters in interviews

- "LoRA vs QLoRA — what exactly is the difference?" The one-line answer (same algorithm, 4-bit frozen base) signals
  you have run one; the three ideas (NF4, double quantization, paged optimizers) are the follow-up.
- Memory arithmetic is the real interview question: "can we fine-tune a 7B/70B model on one 24/80 GB GPU?" You need
  bytes per parameter for weights, gradients and optimizer states, from memory.
- Knowing what `prepare_model_for_kbit_training` and `bnb_4bit_compute_dtype` do separates "copied a Colab" from
  "understands the recipe".

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `tn05` | LoRA vs QLoRA — what exactly is the difference? |
| `tn06` | When would you choose full fine-tuning over LoRA? (the 16 bytes/param arithmetic) |
| `tn31` | What does it cost, roughly, to fine-tune a model? |
| `tn35` | What is gradient checkpointing and why would you use it? |
| `tn36` | What is mixed-precision training and what is bf16 for? |
| `fo30` | What is quantisation, in one paragraph, and what does it cost you? |
| `in07` | What is quantisation and which method would you pick? |
| `tr42` | The memory footprint of this 3B model is 12.9 GB. Why, and how would you cut it? |
| `tn23` | What is ZeRO / FSDP? (the optimizer-state arithmetic) |
| `tn41` | What are the five key hyperparameters of a QLoRA setup, and what does each one control? |
| `tn42` | Which quantization settings do you choose for QLoRA, and what does each one cost? |
| `tn43` | What are the five key training hyperparameters for a LoRA or QLoRA fine-tune? |
| `tn44` | Which optimizer do you use for LoRA and QLoRA, and what do the 8-bit and paged variants change? |
| `tn45` | Your QLoRA run is misbehaving. For each symptom, which hyperparameter do you check first? |
| `tn46` | What are the four steps of training, and where does each hyperparameter act? |

In [ ]:
import json, time
import numpy as np
import torch
import matplotlib.pyplot as plt

torch.manual_seed(0)
t_start = time.time()
import bitsandbytes as bnb
import bitsandbytes.functional as bnbF
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_ID = "HuggingFaceTB/SmolLM2-135M-Instruct"
print(f"bitsandbytes {bnb.__version__}; imports took {time.time() - t_start:.1f}s")

## 1. QLoRA's three ideas

**In plain English:** the frozen base model is the big memory cost, and it never changes during LoRA training — so
store it in 4 bits. Gradients still flow *through* it (de-quantised on the fly) into small full-precision adapters.

| Idea | What it does | Saves |
|---|---|---|
| **4-bit NormalFloat (NF4)** | 16 quantisation levels placed at quantiles of a normal distribution — where pretrained weights actually are — with one scale (`absmax`) per block of 64 weights | ~4× vs bf16 for the base weights |
| **Double quantization** | the per-block fp32 scales are themselves quantised to 8 bits (in blocks of 256) | 0.5 → ~0.127 bits/parameter of overhead (~3 GB on a 65B model) |
| **Paged optimizers** | optimizer states live in CUDA unified memory and page to CPU RAM when a long sequence spikes GPU memory | survives memory spikes instead of OOM-crashing |

The algorithm on top is unchanged LoRA. The quantisation error of the base is a *fixed* bias the adapters can learn
around; it does not compound, because the base never updates.

### The NF4 code book

In [ ]:
nf4 = bnbF.get_4bit_type("nf4", device="cpu")
print("NF4 levels:", [round(v, 3) for v in nf4.tolist()])
w = torch.randn(512, 512) * 0.02                           # weight-like values: roughly normal, small
fig, ax = plt.subplots(figsize=(9, 3.2))
scaled = (w.flatten() / w.abs().max()).numpy()
ax.hist(scaled, bins=120, density=True, color="#adb5bd", label="weights / absmax (normal-ish)")
for v in nf4.tolist():
    ax.axvline(v, color="#3b5bdb", lw=1)
for v in np.linspace(-1, 1, 16):
    ax.axvline(v, color="#e8590c", lw=1, ls=":", ymax=0.25)
ax.set_title("NF4 levels (blue) are dense where weights are; uniform int4 levels (orange, dotted) are not")
ax.set_xlabel("normalised weight value"); ax.legend(loc="upper left", fontsize=8); plt.show()

errors = {}
for qt in ["nf4", "fp4"]:
    q, state = bnbF.quantize_4bit(w, blocksize=64, quant_type=qt)
    errors[qt] = (bnbF.dequantize_4bit(q, state) - w).abs().mean().item()
print(f"mean |error| per weight — NF4: {errors['nf4']:.2e}   FP4: {errors['fp4']:.2e}")
assert errors["nf4"] < errors["fp4"]

### Double quantization, in bits per parameter

In [ ]:
block, block2 = 64, 256
plain = 32 / block                                  # one fp32 absmax per 64 weights
double = 8 / block + 32 / (block * block2)          # 8-bit absmax per 64, plus one fp32 scale per 256 of those
q, st = bnbF.quantize_4bit(w, blocksize=64, quant_type="nf4", compress_statistics=True)
print(f"scale overhead: {plain:.3f} bits/param without, {double:.3f} with double quantization")
print(f"bitsandbytes agrees: nested={st.nested}, absmax dtype {st.absmax.dtype}, second-level blocksize {st.state2.blocksize}")
print(f"saving on a 65B model: {65e9 * (plain - double) / 8 / 1e9:.1f} GB")

## 2. Load SmolLM2 in 4-bit NF4

`bnb_4bit_compute_dtype` is the dtype the de-quantised weights are *computed* in. On a GPU you would use
`torch.bfloat16`; on this CPU we use `torch.float32`.

In [ ]:
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
                                bnb_4bit_compute_dtype=torch.float32)
try:
    tok = AutoTokenizer.from_pretrained(MODEL_ID)
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_config, device_map="cpu")
except Exception as e:
    raise RuntimeError("4-bit loading failed: needs bitsandbytes >= 0.45 with CPU support and the cached model.") from e

n_params = sum(p.numel() for p in model.parameters())
q_proj = model.model.layers[0].self_attn.q_proj
print(f"q_proj is now {type(q_proj).__module__}.{type(q_proj).__name__}; stored weight {tuple(q_proj.weight.shape)} {q_proj.weight.dtype}")
print(f"  = {576 * 576:,} weights packed two per byte into {q_proj.weight.numel():,} bytes; quant_type={q_proj.weight.quant_state.quant_type}")
by_dtype = {}
for p in model.parameters():
    by_dtype[str(p.dtype)] = by_dtype.get(str(p.dtype), 0) + p.numel() * p.element_size()
print("bytes by storage dtype:", {k: f"{v / 1e6:.1f} MB" for k, v in by_dtype.items()})
fp32_mb = 134_515_008 * 4 / 1e6                       # the unquantised parameter count x 4 bytes
foot_4bit = model.get_memory_footprint() / 1e6
print(f"memory footprint: 4-bit {foot_4bit:.0f} MB vs fp32 {fp32_mb:.0f} MB")

Only the Linear layers inside the decoder blocks are quantised. The embedding matrix (49152 × 576 = 28M
parameters, tied with `lm_head`) stays in 16-bit — for this tiny model it is a big share of what is left, while for a
7B model the embeddings are ~2% of the weights, so 4-bit gets much closer to the ideal 4× (or 8× vs fp32) there.

## 3. Prepare for k-bit training, attach LoRA

`prepare_model_for_kbit_training` freezes the base, casts the remaining non-quantised parameters (norms,
embeddings) to fp32 for stable training, makes the input embeddings emit gradients (so the adapters in layer 0 get
a signal even though the embedding is frozen) and, by default, turns on gradient checkpointing. We switch
checkpointing off: it trades ~20–30% extra compute for activation memory, which a 135M model on a CPU does not need.

In [ ]:
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=False)
foot_prepared = model.get_memory_footprint() / 1e6
print(f"footprint after prepare: {foot_prepared:.0f} MB (norms/embeddings upcast to fp32)")

lora_cfg = LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05, target_modules=["q_proj", "v_proj"], task_type="CAUSAL_LM")
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()
adapter = model.base_model.model.model.layers[0].self_attn.q_proj
print("adapter dtype:", adapter.lora_A["default"].weight.dtype, "| base layer:", type(adapter.base_layer).__name__)
assert isinstance(adapter.base_layer, bnb.nn.Linear4bit) and adapter.lora_A["default"].weight.dtype == torch.float32

The base layer is still a 4-bit `Linear4bit`; `lora_A` / `lora_B` are ordinary fp32 matrices. Every forward pass
de-quantises the 4-bit weight block by block, adds the LoRA path, and backpropagates *through* the de-quantised weight
into the adapters.

## 4. Train a few steps on the same tiny task

The JSON-answer behaviour from notebook 04, 16 examples. Optimizer: bitsandbytes' `PagedAdamW8bit` — the QLoRA
paper's choice. On a CPU, "paged" is a no-op (there is no GPU memory to page from), but the 8-bit state and the API
are the real ones.

In [ ]:
SYSTEM = "You are a helpful assistant."
train_qa = [("What is the capital of France?", "Paris"), ("What is 2 + 2?", "4"), ("How many legs does a spider have?", "8"),
            ("What is the largest planet?", "Jupiter"), ("Who wrote Romeo and Juliet?", "William Shakespeare"),
            ("What language is spoken in Spain?", "Spanish"), ("How many days are in a week?", "7"),
            ("What is the capital of Japan?", "Tokyo"), ("What animal says moo?", "A cow"), ("What is 10 minus 3?", "7"),
            ("Which planet is called the red planet?", "Mars"), ("What do bees make?", "Honey"),
            ("What is frozen water called?", "Ice"), ("What is the capital of Italy?", "Rome"),
            ("What is 5 times 6?", "30"), ("Which ocean is the largest?", "The Pacific Ocean")]
heldout_q = ["What is the capital of Spain?", "What color is grass?", "What is 3 plus 5?", "Who painted the Mona Lisa?"]

def build(question, answer):
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": question}]
    prompt = tok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=True, return_dict=True)["input_ids"]
    full = tok.apply_chat_template(msgs + [{"role": "assistant", "content": json.dumps({"answer": answer})}],
                                   tokenize=True, return_dict=True)["input_ids"]
    full = full[: full.index(tok.eos_token_id, len(prompt)) + 1]
    return full, [-100] * len(prompt) + full[len(prompt):]

def collate(rows):
    T = max(len(x) for x, _ in rows)
    batch = {"input_ids": torch.full((len(rows), T), tok.pad_token_id), "labels": torch.full((len(rows), T), -100),
             "attention_mask": torch.zeros((len(rows), T), dtype=torch.long)}
    for i, (x, y) in enumerate(rows):
        batch["input_ids"][i, :len(x)], batch["labels"][i, :len(y)], batch["attention_mask"][i, :len(x)] = torch.tensor(x), torch.tensor(y), 1
    return batch

tok.padding_side = "left"
@torch.no_grad()
def generate(m, questions, max_new_tokens=14):
    enc = tok.apply_chat_template([[{"role": "system", "content": SYSTEM}, {"role": "user", "content": q}] for q in questions],
                                  add_generation_prompt=True, return_tensors="pt", return_dict=True, padding=True)
    out = m.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tok.pad_token_id)
    return [tok.decode(o[enc["input_ids"].shape[1]:], skip_special_tokens=True).strip() for o in out]

def is_json(text):
    try:
        return "answer" in json.loads(text)
    except (ValueError, TypeError):
        return False

model.eval()
with model.disable_adapter():
    before = generate(model, heldout_q)
print("before:", before)

In [ ]:
rows = [build(q, a) for q, a in train_qa]
optimizer = bnb.optim.PagedAdamW8bit([p for p in model.parameters() if p.requires_grad], lr=2e-3)
g = torch.Generator().manual_seed(0)
losses = []
t0 = time.time()
try:
    model.train()
    for step in range(1, 9):
        idx = torch.randperm(len(rows), generator=g)[:4].tolist()
        loss = model(**collate([rows[i] for i in idx])).loss
        loss.backward(); optimizer.step(); optimizer.zero_grad()
        losses.append(loss.item())
        print(f"step {step}: loss {loss.item():.3f}")
    trained_in_4bit = True
except (RuntimeError, NotImplementedError) as e:          # older bitsandbytes builds had no CPU backward
    trained_in_4bit = False
    print("4-bit training is not supported by this bitsandbytes build:", str(e)[:200])
model.eval()
print(f"trained on a 4-bit base: {trained_in_4bit}; 8 steps in {time.time() - t0:.1f}s")
assert trained_in_4bit and losses[-1] < losses[0] / 2

In [ ]:
plt.figure(figsize=(7, 3.2))
plt.plot(range(1, len(losses) + 1), losses, "o-")
plt.xlabel("step"); plt.ylabel("loss on answer tokens"); plt.yscale("log")
plt.title("QLoRA on CPU: LoRA adapters learning through a 4-bit NF4 base"); plt.show()

In [ ]:
after = generate(model, heldout_q)
for q, b, a in zip(heldout_q, before, after):
    print(f"Q: {q}\n   before: {b!r}\n   after : {a!r}")
rate_before, rate_after = np.mean([is_json(b) for b in before]), np.mean([is_json(a) for a in after])
print(f"\nJSON answers: before {rate_before:.0%} -> after {rate_after:.0%}")
assert rate_after > rate_before

Same behaviour change as the full-precision LoRA in notebook 04, from a base stored in 4 bits. Now read the
*content*: the format is perfect, but some answers the base got right in prose may now be wrong inside the JSON.
Eight high-learning-rate steps on 16 examples teach the format and also disturb what the model knew — a small-scale
case of forgetting ([07](07_distillation_and_catastrophic_forgetting.ipynb)). Score content, not just format. (If a
bitsandbytes build cannot backpropagate through 4-bit layers on your hardware, the cell above says so; the fallback
is the fp32-base LoRA of notebook 04 — same adapters, more memory.)

## 5. Memory: what actually fills the GPU

For a model with N parameters, per parameter (activations excluded — they depend on batch × sequence length, and
gradient checkpointing cuts them):

| | weights | gradients | optimizer (Adam m, v) | master copy | total / param |
|---|---|---|---|---|---|
| full FT, mixed precision | 2 B (bf16) | 2 B | 8 B (fp32) | 4 B (fp32) | **~16 B** |
| LoRA | 2 B (bf16, frozen) | only adapters | only adapters | only adapters | **~2 B** + adapters |
| QLoRA | ~0.52 B (NF4 + scales) | only adapters | only adapters | only adapters | **~0.5 B** + adapters |

First the measured numbers for our 135M model, then the arithmetic for 7B:

In [ ]:
lora_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
adapter_train_mb = lora_params * (4 + 4 + 8) / 1e6          # fp32 weights + grads + Adam states for the adapters
full_ft_mb = 134_515_008 * 16 / 1e6
measured = {"fp32 model\n(inference)": fp32_mb, "full FT\n(16 B/param)": full_ft_mb,
            "4-bit base\n(as loaded)": foot_4bit, "4-bit base + LoRA\ntraining state": foot_prepared + adapter_train_mb}
for k, v in measured.items():
    print(f"{k.replace(chr(10), ' '):34s} {v:8.0f} MB")

fig, ax = plt.subplots(figsize=(8, 3.4))
ax.bar(list(measured), list(measured.values()), color=["#adb5bd", "#e8590c", "#74c0fc", "#2f9e44"])
ax.set_ylabel("MB (weights + training state, no activations)"); ax.set_title("SmolLM2-135M: memory by setup")
plt.show()

In [ ]:
N, r, L, d, d_ff = 6.74e9, 16, 32, 4096, 11008
lora_n = L * r * (4 * (d + d) + 2 * (d + d_ff) + (d_ff + d))            # all-linear adapters, r=16
GB = 1e9
setups = {
    "full FT (bf16 + Adam)": {"weights": N * 2, "gradients": N * 2, "optimizer + master": N * 12},
    "LoRA (bf16 base)":      {"weights": N * 2 + lora_n * 4, "gradients": lora_n * 4, "optimizer + master": lora_n * 8},
    "QLoRA (NF4 base)":      {"weights": N * (4 + double) / 8 + lora_n * 4, "gradients": lora_n * 4, "optimizer + master": lora_n * 8},
}
fig, ax = plt.subplots(figsize=(9, 3.6))
bottom = np.zeros(len(setups))
for part, color in [("weights", "#3b5bdb"), ("gradients", "#e8590c"), ("optimizer + master", "#fab005")]:
    vals = np.array([s[part] / GB for s in setups.values()])
    ax.bar(list(setups), vals, bottom=bottom, label=part, color=color); bottom += vals
for i, total in enumerate(bottom):
    ax.text(i, total + 2, f"{total:.1f} GB", ha="center")
for gpu, y in [("24 GB GPU", 24), ("80 GB GPU", 80)]:
    ax.axhline(y, color="grey", ls=":"); ax.text(2.45, y + 1, gpu, fontsize=8, ha="right")
ax.set_ylabel("GB (before activations)"); ax.set_title(f"A 7B model: why QLoRA fits on one consumer GPU (LoRA r=16: {lora_n / 1e6:.0f}M params)")
ax.legend(); plt.show()
for name, s in setups.items():
    print(f"{name:24s}: {sum(s.values()) / GB:6.1f} GB")
assert sum(setups["full FT (bf16 + Adam)"].values()) / GB > 100 > sum(setups["LoRA (bf16 base)"].values()) / GB

~108 GB for a full fine-tune (hence multi-GPU + ZeRO/FSDP, `tn23`), ~14 GB for LoRA, ~4 GB for QLoRA — plus
activations, which is where gradient checkpointing, shorter sequences and smaller micro-batches come in. The ordered
list to turn when training OOMs (`tn35`): gradient accumulation (smaller micro-batch), gradient checkpointing, ZeRO
stage, LoRA instead of full, then quantise the base (QLoRA).

## 6. The standard GPU recipe (for reference — needs a CUDA GPU)

```python
# pip install transformers peft bitsandbytes accelerate trl    (CUDA GPU required for bf16 + paged optimizers)
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16)
model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.1-8B-Instruct",
                                             quantization_config=bnb_config, device_map="auto")
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model = get_peft_model(model, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05,
                                         target_modules="all-linear", task_type="CAUSAL_LM"))
args = TrainingArguments(output_dir="out", bf16=True, optim="paged_adamw_8bit", learning_rate=2e-4,
                         per_device_train_batch_size=4, gradient_accumulation_steps=4, num_train_epochs=2,
                         lr_scheduler_type="cosine", warmup_ratio=0.03, gradient_checkpointing=True, logging_steps=10)
# then Trainer(model=model, args=args, train_dataset=..., data_collator=...) or TRL's SFTTrainer
```

After training, either serve base-in-4-bit + adapter, or load the base in bf16, attach the adapter and
`merge_and_unload()` — merging into a *4-bit* base would re-quantise the merged weights and lose precision.

## 7. LoRA vs QLoRA

| | LoRA | QLoRA |
|---|---|---|
| frozen base stored in | bf16 / fp16 (2 B/param) | NF4 4-bit + double-quantised scales (~0.52 B/param) |
| adapters | bf16/fp32, trainable | same |
| algorithm | LoRA | **the same** LoRA |
| memory for 7B (before activations) | ~14 GB | ~4 GB |
| speed | faster | slower: de-quantise every forward and backward (often 20–40% slower) |
| quality | reference | usually within noise of LoRA (the paper matched 16-bit fine-tuning); the base's quantisation error is fixed |
| use when | the bf16 base fits | it does not — QLoRA is the reason you can train at all |

## 8. The ten knobs of a QLoRA run, measured

Everything you set on a QLoRA run is one of ten hyperparameters. **Five QLoRA knobs** decide *what* gets trained and
how the frozen base is stored; **five training knobs** decide *how* the run goes. Each one acts inside one of the
four steps every training step performs:

| Knob | Where it lives | Acts in step | Typical |
|---|---|---|---|
| **target modules** | `LoraConfig(target_modules=...)` | forward, backward | `"all-linear"` or q, k, v, o |
| **r** | `LoraConfig(r=...)` | backward (how many parameters get gradients) | 16 (8–64) |
| **alpha** | `LoraConfig(lora_alpha=...)` | forward: output × α/r | α = 2r |
| **quantization** | `BitsAndBytesConfig(...)` | forward: base de-quantised on the fly | 4-bit NF4 + double quant |
| **dropout** | `LoraConfig(lora_dropout=...)` | forward, training only | 0.05–0.1 |
| **epochs** | `num_train_epochs` | how often the whole loop runs | 1–3 |
| **batch size** | `per_device_train_batch_size` | forward: examples per pass | 1–8 per GPU |
| **learning rate** | `learning_rate` (+ scheduler, warmup) | optimization: step size | 2e-4, cosine, 3% warmup |
| **gradient accumulation** | `gradient_accumulation_steps` | loss ÷ N, backward sums N micro-batches | effective batch 16–64 |
| **optimizer** | `optim` | optimization | `paged_adamw_8bit` / `paged_adamw_32bit` |

The cells below measure each one on the SmolLM2 model already loaded above (4-bit base, LoRA r=8, α=16, dropout 0.05
on `q_proj` and `v_proj`, trained for 8 steps in section 4). The interactive version, with a config builder that
reviews your settings, is the "ten knobs" panel in [genai_flow chapter 21](../../genai_flow/index.html).

### The four steps of training, on the QLoRA model

In [ ]:
# ---- One QLoRA training step, taken apart into the four steps of training ----
# Training = tweaking the parameters on training data, in a way that should generalise to unseen data.
# Every step, on every batch, is these same four operations. Watch what each one touches.
lora_A = adapter.lora_A["default"].weight          # adapter = layer 0's q_proj, wrapped by PEFT in section 3
lora_B = adapter.lora_B["default"].weight
base_w = adapter.base_layer.weight                 # the frozen base weight: 4-bit, packed two per byte (uint8)
B_before, base_before = lora_B.detach().clone(), base_w.detach().clone()

trainable = [p for p in model.parameters() if p.requires_grad]
step_opt = torch.optim.AdamW(trainable, lr=1e-4)    # a fresh optimizer just for this one demonstration step
batch = collate(rows[:4])                            # knob: batch size = 4 examples

model.eval()                                         # measure the loss before, without dropout noise
with torch.no_grad():
    loss_before = model(**batch).loss.item()
model.train()                                        # training mode: lora_dropout is active from here

# 1. FORWARD PASS: predict the output given the inputs.
#    Every 4-bit base Linear is de-quantised block by block into the compute dtype (knob: quantization), and each
#    TARGETED Linear adds (alpha / r) * B(A(dropout(x))) on top (knobs: target modules, r, alpha, dropout).
out = model(**batch)

# 2. LOSS CALCULATION: how different was the prediction from the ground truth?
#    Cross-entropy on the answer tokens only: the prompt positions are labelled -100 by build() and ignored.
loss = out.loss
print(f"1-2. forward pass + loss      : {loss.item():.4f}")

# 3. BACKWARD PASS: which way should each parameter move? (the gradients)
#    Gradients flow back THROUGH the frozen, de-quantised base INTO the adapters. The base itself gets none.
loss.backward()
n_grad = sum(p.grad is not None for p in model.parameters())
print(f"3.   backward pass            : {n_grad} tensors got a gradient, {len(trainable)} are trainable LoRA A/B matrices")
print(f"     4-bit base has a gradient: {base_w.grad is not None}")

# 4. OPTIMIZATION: update the parameters a tiny step, to do better next time (knobs: learning rate, optimizer).
step_opt.step()
step_opt.zero_grad()                                 # clear the gradients, or the next backward() adds to them
model.eval()
with torch.no_grad():
    loss_after = model(**batch).loss.item()
print(f"4.   optimizer step           : lora_B moved by up to {(lora_B - B_before).abs().max():.1e}, "
      f"the base by {(base_w.int() - base_before.int()).abs().max()}; loss {loss_before:.4f} -> {loss_after:.4f}")

assert n_grad == len(trainable) and base_w.grad is None      # only adapters learn
assert torch.equal(base_w, base_before) and not torch.equal(lora_B, B_before)
assert loss_after < loss_before

### Knobs 1 and 2 — target modules and r: how many parameters train

Each targeted `nn.Linear` of shape in → out gets A (r × in) and B (out × r): **r × (in + out)** parameters, in every
layer. So the count is linear in r, and the modules you pick decide the constant.

In [ ]:
# ---- target_modules and r decide the trainable parameter count ----
def lora_param_count(m, r, targets):
    """r * (in + out) for every Linear whose name is in `targets`, read from the model's real shapes."""
    total = 0
    for name, mod in m.named_modules():
        # the PEFT wrapper and plain Linear4bit layers both expose in_features / out_features;
        # skip the wrapper's own children (base_layer, lora_A.default, lora_B.default)
        if name.split(".")[-1] in targets and hasattr(mod, "in_features"):
            total += r * (mod.in_features + mod.out_features)
    return total

# check the formula against what PEFT actually made trainable (r=8 on q_proj and v_proj)
current = lora_param_count(model, 8, ["q_proj", "v_proj"])
print(f"formula {current:,} vs PEFT {lora_params:,} trainable parameters")
assert current == lora_params

TARGETS = {
    "q, v (LoRA paper)":        ["q_proj", "v_proj"],
    "q, k, v, o (attention)":   ["q_proj", "k_proj", "v_proj", "o_proj"],
    "all-linear (QLoRA paper)": ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
}
print(f"\n{'target modules':26s}" + "".join(f"{'r=' + str(r):>12s}" for r in (4, 8, 16, 32, 64)))
for label, t in TARGETS.items():
    counts = [lora_param_count(model, r, t) for r in (4, 8, 16, 32, 64)]
    print(f"{label:26s}" + "".join(f"{c:>12,}" for c in counts))
    assert counts[1] == 2 * counts[0] and counts[4] == 16 * counts[0]    # linear in r
print(f"\nall-linear at r=16 is {lora_param_count(model, 16, TARGETS['all-linear (QLoRA paper)']) / 134_515_008:.2%} "
      f"of the 134.5M-parameter model")
# The MLP (gate/up/down) is most of each layer's parameters, which is why adding it costs more but usually fine-tunes better.

### Knob 3 — alpha: the adapter's volume

PEFT multiplies the adapter output by **α / r** (`adapter.scaling`). The ratio is what matters, and it is the trap: raise
r without raising α and the adapter silently gets quieter.

In [ ]:
# ---- lora_alpha: the layer's output is W x + (alpha / r) * B A x ----
model.eval()                                         # eval mode: dropout off, so the output is deterministic
print("adapter.scaling =", adapter.scaling, " (lora_alpha 16 / r 8)")
x = torch.randn(3, 576)                              # three fake hidden states of the model's width, 576
with torch.no_grad():
    bax = adapter.lora_B["default"](adapter.lora_A["default"](x))    # B A x, before scaling
    added = adapter(x) - adapter.base_layer(x)                        # what the wrapped layer actually adds to W x
print(f"what the layer adds / (B A x) = {(added / bax).median():.3f}")
assert torch.allclose(added, bax * adapter.scaling["default"], atol=1e-5)

# The trap: keep lora_alpha fixed while sweeping r, and the scale changes under you.
print("\n   r   alpha=16 fixed   alpha=2r")
for r in (8, 16, 32, 64):
    print(f"{r:4d}   scale {16 / r:6.3f}     scale {2 * r / r:.1f}")
# Move alpha WITH r (alpha = 2r keeps the scale at 2), or use_rslora=True, which scales by alpha / sqrt(r).

### Knob 5 — dropout: only on the adapter's input, only in training

`lora_dropout` randomly zeroes part of the adapter's *input* during training: the layer computes
`W x + (α/r) · B A dropout(x)`. The frozen path is never touched, and `model.eval()` switches it off.

In [ ]:
# ---- lora_dropout: random in train mode, gone in eval mode, never on the frozen path ----
print(adapter.lora_dropout["default"])               # Dropout(p=0.05)
with torch.no_grad():
    adapter.train()                                  # training mode for this one layer: dropout active
    t1, t2 = adapter(x), adapter(x)
    adapter.eval()                                   # inference mode: dropout is the identity
    e1, e2 = adapter(x), adapter(x)
    base1, base2 = adapter.base_layer(x), adapter.base_layer(x)
print(f"train mode, same input twice: max difference {(t1 - t2).abs().max():.2e}  (dropout at work)")
print(f"eval mode,  same input twice: max difference {(e1 - e2).abs().max():.2e}")
assert not torch.equal(t1, t2) and torch.equal(e1, e2) and torch.equal(base1, base2)
# A regulariser: it matters on small datasets like ours (16 examples) and hardly at all on large ones.

### Knob 4 — quantization: what the frozen base costs

Sections 1–2 measured NF4 on real weights. Here is the same choice as memory for a real 8B model, using Llama 3 8B's
config shapes. Only the decoder Linears are quantised; embeddings, the untied `lm_head` and the norms stay 16-bit.

In [ ]:
# ---- quantization: bits per base weight, including the per-block scales ----
d, kv, ff, layers, vocab = 4096, 1024, 14336, 32, 128256          # Llama 3 8B, from its config.json
linear = layers * (2 * d * d + 2 * d * kv + 3 * d * ff)           # q, o + k, v (grouped-query) + gate, up, down
rest = 2 * vocab * d + layers * 2 * d + d                         # embed_tokens + lm_head + norms: kept 16-bit
print(f"Llama 3 8B: {linear + rest:,} parameters, {linear / (linear + rest):.0%} of them in quantisable Linears")

BITS = {
    "bf16 (plain LoRA)":   16,
    "8-bit":               8,
    "4-bit NF4":           4 + 32 / 64,                   # one fp32 absmax per 64 weights: +0.5 bit
    "4-bit NF4 + double":  4 + 8 / 64 + 32 / (64 * 256),  # absmax stored in 8 bits, + one fp32 per 256 of them
}
gb = {name: (linear * bits / 8 + rest * 2) / 1e9 for name, bits in BITS.items()}
for name, g in gb.items():
    print(f"  {name:20s} {BITS[name]:6.3f} bits/weight   frozen base {g:5.1f} GB")
assert linear + rest == 8_030_261_248
assert list(gb.values()) == sorted(gb.values(), reverse=True)            # every step down saves memory
assert f"{gb['4-bit NF4 + double']:.1f}" == "5.7"
# Adapters are never quantised: they train in 16/32-bit on top. Skip quantization entirely if the bf16 model fits:
# it is faster per step and the reference quality.

### The five training knobs

**Epochs, batch size and gradient accumulation** decide how many optimizer steps you get; **learning rate** sets the
size of each one (with a schedule); the **optimizer** decides how that step is taken and what state it keeps.
The Trainer's arithmetic, then the learning-rate schedule it lays over those steps:

In [ ]:
# ---- epochs x batch size x gradient accumulation = optimizer steps; the LR schedule is laid over them ----
import math
from transformers import get_cosine_schedule_with_warmup

def optimizer_steps(n_examples, batch_size, grad_accum, epochs, n_gpus=1):
    micro_batches = math.ceil(n_examples / (batch_size * n_gpus))   # forward/backward passes per epoch, per GPU
    per_epoch = math.ceil(micro_batches / grad_accum)               # one optimizer.step() per `grad_accum` of them
    return batch_size * grad_accum * n_gpus, per_epoch, per_epoch * epochs

eff, per_epoch, total = optimizer_steps(n_examples=5000, batch_size=4, grad_accum=4, epochs=2)
print(f"5,000 examples, batch 4, accumulation 4, 2 epochs -> effective batch {eff}, "
      f"{per_epoch} steps per epoch, {total} optimizer steps")
assert (eff, per_epoch, total) == (16, 313, 626)

# learning_rate=2e-4, lr_scheduler_type="cosine", warmup_ratio=0.03: exactly what the Trainer builds
dummy = torch.nn.Parameter(torch.zeros(1))
opt = torch.optim.AdamW([dummy], lr=2e-4)
warmup = math.ceil(0.03 * total)
sched = get_cosine_schedule_with_warmup(opt, num_warmup_steps=warmup, num_training_steps=total)
lrs = []
for _ in range(total):
    lrs.append(opt.param_groups[0]["lr"])
    opt.step(); sched.step()

fig, ax = plt.subplots(figsize=(8, 2.8))
ax.plot(lrs); ax.axvline(warmup, color="grey", ls=":")
ax.set(title=f"learning rate: {warmup} warmup steps, then cosine decay over {total} steps", xlabel="optimizer step", ylabel="lr")
plt.show()
assert lrs[0] == 0 and max(lrs) == lrs[warmup] == 2e-4 and lrs[-1] < 2e-6
# Warmup protects the freshly initialised adapters from full-size steps; decay lets the run settle at the end.

### Gradient accumulation really is a bigger batch

Accumulating N micro-batches, each loss divided by N, gives exactly the gradient of one batch N times bigger, at the
memory cost of one micro-batch. Exactly, as long as every example carries the same weight:

In [ ]:
# ---- gradient accumulation: 4 micro-batches of 2 == 1 batch of 8 ----
torch.manual_seed(0)
Xa, ya = torch.randn(8, 5, dtype=torch.float64), torch.randn(8, dtype=torch.float64)
w = torch.zeros(5, dtype=torch.float64, requires_grad=True)

((Xa @ w - ya) ** 2).mean().backward()               # one batch of 8
g_big = w.grad.clone(); w.grad = None

for i in range(0, 8, 2):                              # 4 micro-batches of 2
    micro_loss = ((Xa[i:i + 2] @ w - ya[i:i + 2]) ** 2).mean()
    (micro_loss / 4).backward()                       # divide by N; .grad ADDS across backward() calls
print(f"max |difference| between the two gradients: {(w.grad - g_big).abs().max():.1e}")
assert torch.allclose(w.grad, g_big)

# The LLM gotcha: the loss is a mean over TOKENS, and micro-batches hold different numbers of answer tokens.
token_losses = torch.rand(8, dtype=torch.float64)
short, long = token_losses[:2], token_losses[2:]      # one micro-batch with 2 answer tokens, one with 6
mean_of_means = (short.mean() + long.mean()) / 2      # naive accumulation: each micro-batch weighted equally
print(f"true token mean {token_losses.mean():.4f} vs mean of micro-batch means {mean_of_means:.4f}")
assert not torch.isclose(token_losses.mean(), mean_of_means)
# Recent transformers releases fix this by dividing by the token count of the WHOLE accumulated batch.

### The optimizer: what its state costs

AdamW keeps two running averages per trainable parameter. Section 4 trained with bitsandbytes' `PagedAdamW8bit`;
measure its state against a plain fp32 `AdamW` on the same LoRA parameters:

In [ ]:
# ---- optimizer state per trainable parameter: 8-bit vs fp32 AdamW ----
def state_bytes(opt):
    return sum(t.numel() * t.element_size() for s in opt.state.values() for t in s.values() if torch.is_tensor(t))

adamw = torch.optim.AdamW(trainable, lr=0.0)          # lr 0 so this measurement cannot change the model
for p_ in trainable:
    p_.grad = torch.zeros_like(p_)
adamw.step(); adamw.zero_grad()                       # state tensors are created on the first step

b8, b32 = state_bytes(optimizer), state_bytes(adamw)
print(f"{lora_params:,} LoRA parameters")
print(f"  PagedAdamW8bit state: {b8 / 1e6:.2f} MB  ({b8 / lora_params:.1f} bytes per parameter)")
print(f"  fp32 AdamW state    : {b32 / 1e6:.2f} MB  ({b32 / lora_params:.1f} bytes per parameter)")
assert b8 < 0.5 * b32
# (Tensors under 4,096 elements, such as the small v_proj lora_B, keep 32-bit state even in the 8-bit optimizer.)

# At scale the real lesson: with adapters the optimizer is small; with full fine-tuning it is enormous.
lora_8b = 32 * 16 * (2 * (4096 + 4096) + 2 * (4096 + 1024) + 3 * (4096 + 14336))    # r=16, all-linear, Llama 3 8B
print(f"\nLlama 3 8B, LoRA r=16 all-linear ({lora_8b / 1e6:.1f}M params): AdamW state {lora_8b * 8 / 1e9:.2f} GB, "
      f"8-bit {lora_8b * 2 / 1e9:.2f} GB")
print(f"Llama 3 8B, FULL fine-tune (8.03B params): AdamW state {8.03e9 * 8 / 1e9:.0f} GB")
assert lora_8b == 41_943_040
# So the 4-bit base, not the 8-bit optimizer, is what makes QLoRA fit. "paged_" is about surviving memory spikes.

### Epochs, and reading the symptoms

Epochs are the one knob this notebook cannot show honestly with 16 examples: the train-versus-validation curves in
[06_deep_learning/02 · training loop](../06_deep_learning/02_training_loop_optimizers_lr_schedules.ipynb) show the
turn where more epochs start to overfit, and [07 · forgetting](07_distillation_and_catastrophic_forgetting.ipynb)
shows what too many high-LR steps do to what the model already knew. Keep the checkpoint with the best *eval* loss.

When a run misbehaves, each symptom points at one knob first (`tn45`):

| Symptom | First knob | Do |
|---|---|---|
| loss flat from step one | learning rate, then α/r | LR up to ~2e-4; keep α ≈ 2r |
| loss spikes or NaN | learning rate | lower it, add warmup, clip gradients |
| train loss falls, eval loss rises | epochs | fewer epochs or more data; dropout 0.1; lower r |
| learns the format, not the task | target modules, r | add gate/up/down; raise r |
| OOM loading the model | quantization | 4-bit NF4 + double quant |
| OOM during training | batch size | halve batch, double accumulation; gradient checkpointing; paged optimizer |
| noisy, run-to-run variance | effective batch | raise gradient accumulation; fix the seed |

## Try it yourself

**1.** How much GPU memory would QLoRA need for a **70B** model (weights + LoRA r=16 training state, no activations)?
Assume 80 layers, hidden 8192, MLP 28672.

In [ ]:
# Solution — try it yourself first, then run this
N70, L70, d70, f70 = 70.6e9, 80, 8192, 28672
lora70 = L70 * 16 * (2 * (d70 + d70) + 2 * (d70 + 1024) + 2 * (d70 + f70) + (f70 + d70))   # q,o full; k,v GQA (1024 wide)
total = N70 * (4 + double) / 8 + lora70 * 16
print(f"LoRA params: {lora70 / 1e6:.0f}M   QLoRA weights + adapter state: {total / 1e9:.1f} GB -> fits one 80 GB GPU with room for activations")

**2.** Quantise a weight matrix that has a few large **outliers** with NF4 and compare the error with and without
them. Why does blockwise quantization (one `absmax` per 64 weights) help?

In [ ]:
# Solution — try it yourself first, then run this
torch.manual_seed(1)
w2 = torch.randn(256, 256) * 0.02
w2_out = w2.clone(); w2_out[0, :4] = 1.0                           # four outliers in one block
for name, mat in [("no outliers", w2), ("with outliers", w2_out)]:
    for bs in (64, 4096):
        q_, s_ = bnbF.quantize_4bit(mat, blocksize=bs, quant_type="nf4")
        print(f"{name:14s} blocksize {bs:4d}: mean |error| {(bnbF.dequantize_4bit(q_, s_) - mat).abs().mean():.2e}")
print("An outlier inflates the absmax of its block only; with small blocks the damage stays local.")

**3.** Why is `bnb_4bit_compute_dtype=torch.float32` a bad choice on a GPU, and what is bf16's advantage over fp16
for training? (Answer in words.)

In [ ]:
# Solution — try it yourself first, then run this
print("fp32 compute doubles activation memory and misses the tensor-core speed-up; bf16 is the GPU default.")
print("bf16 keeps fp32's 8-bit exponent (same range), so no loss scaling is needed; fp16 has more mantissa but")
print("a narrow range that overflows/underflows in training. Range, not precision, is why bf16 won (tn36).")
print(f"notebook runtime: {time.time() - t_start:.0f}s")

## Say it in an interview

- **LoRA vs QLoRA in one line:** "Same algorithm; QLoRA stores the frozen base in 4-bit NF4 so it fits. Gradients
  flow through the de-quantised weights into full-precision adapters."
- **The three ideas:** NF4 (levels at normal quantiles, blockwise absmax), double quantization (quantise the scales:
  0.5 → ~0.13 bits/param), paged optimizers (page optimizer state to CPU on memory spikes).
- **Memory numbers:** full FT ≈ 16 B/param (7B → ~110 GB before activations); LoRA ≈ 2 B/param (~14 GB); QLoRA ≈
  0.5 B/param (~4 GB); 70B QLoRA fits one 80 GB GPU.
- **Recipe knobs:** `bnb_4bit_compute_dtype=bfloat16`, `prepare_model_for_kbit_training` (freezes, upcasts norms,
  enables input grads, gradient checkpointing), `optim="paged_adamw_8bit"`, LR ~2e-4, all-linear targets.
- **The ten knobs:** QLoRA — target modules, r, alpha (scale α/r), quantization, dropout; training — epochs,
  batch size, learning rate, gradient accumulation, optimizer. Defaults: all-linear, r=16, α=32, NF4 + double
  quant, dropout 0.05; 1–3 epochs, batch 4 × accumulation 4, LR 2e-4 cosine with 3% warmup, paged AdamW (`tn41`, `tn43`).
- **The four steps:** forward (predict), loss (how wrong), backward (gradients into A and B only), optimization
  (a tiny step). Each knob acts in one of them (`tn46`).
- **Trade-offs:** slower than LoRA (de-quantisation every pass); quality usually close; merge into a bf16 copy of the
  base, not the 4-bit one.
- **Traps:** calling QLoRA "a different algorithm"; forgetting activations in the memory maths; assuming the 4-bit
  file size equals runtime memory (embeddings, norms and the compute dtype add to it — see our 135M model).

## Next

- [06 · DPO preference tuning](06_dpo_preference_tuning.ipynb)
- [04 · LoRA with PEFT](04_lora_with_peft.ipynb) — the full-precision version, save / reload / merge / multi-adapter
- [Quantization with bitsandbytes](../10_huggingface/06_quantization_with_bitsandbytes.ipynb) · [Quantization int8 / int4 / NF4](../07_genai_foundations/07_quantization_int8_int4_nf4.ipynb)
- [KV cache and inference speed](../07_genai_foundations/06_kv_cache_and_inference_speed.ipynb)
- Theory: [genai_flow course](../../genai_flow/index.html) · [interview bank](../../ai_interview_prep/index.html)